# MEDMNIST

## Data Loading

In [1]:
import torch
from src.utils import load_medmnist

def check_medmnist(subset="pathmnist", batch_size=8):
    """
    Load a small MedMNIST subset and print batch shapes and channel info.
    """
    try:
        train_loader, val_loader, test_loader, n_classes, n_channels = load_medmnist(
            batch_size=batch_size,
            subset=subset,
            validation_split=0.2,
            shuffle_dataset=False,
            random_seed=123
        )
    except Exception as e:
        print(f"Error loading MedMNIST subset '{subset}':\n{e}")
        return

    print(f"Successfully loaded MedMNIST subset: '{subset}'")
    print(f" → Number of classes: {n_classes}")
    print(f" → Number of channels: {n_channels}")

    # Grab a single batch from each loader and print shapes
    for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        images, labels = next(iter(loader))
        print(f"\n{name.upper()} batch shape:")
        print(f"  images: {images.shape}     (should be [batch_size, {n_channels}, 28, 28])")
        print(f"  labels: {labels.shape}     (should be [batch_size])")

if __name__ == "__main__":
    # Try a few common subsets:
    for subset in ["pathmnist", "chestmnist", "dermamnist"]:
        check_medmnist(subset=subset, batch_size=8)


Using downloaded and verified file: /home/vito/.medmnist/pathmnist.npz
Using downloaded and verified file: /home/vito/.medmnist/pathmnist.npz
Error loading MedMNIST subset 'pathmnist':
'n_classes'
Using downloaded and verified file: /home/vito/.medmnist/chestmnist.npz
Using downloaded and verified file: /home/vito/.medmnist/chestmnist.npz
Error loading MedMNIST subset 'chestmnist':
'n_classes'
Using downloaded and verified file: /home/vito/.medmnist/dermamnist.npz
Using downloaded and verified file: /home/vito/.medmnist/dermamnist.npz
Error loading MedMNIST subset 'dermamnist':
'n_classes'


## Instantiate a InceptionMnistModel and Forward a Batch

In [2]:
import torch
from src.models import InceptionMNISTModel

def test_forward_on_medmnist(subset="pathmnist", batch_size=4):
    from src.utils import load_medmnist

    # Load a small batch
    train_loader, val_loader, test_loader, n_classes, n_channels = load_medmnist(
        batch_size=batch_size,
        subset=subset,
        validation_split=0.2,
        shuffle_dataset=False,
        random_seed=123
    )
    images, labels = next(iter(train_loader))
    print(f"Loaded one batch of shape: {images.shape} (channels={n_channels}, classes={n_classes})")

    # Define a trivial one-branch Inception block:
    branches = [{
        "depth": 1,
        "filter_sizes": [(3, 3)],
        "filter_channels": [8],
        "use_pooling": False
    }]

    model = InceptionMNISTModel(
        model_params={"branches_params": branches},
        input_channels=n_channels,
        num_classes=n_classes
    )

    # Forward on that batch:
    outputs = model(images)
    print(f"Model output shape: {outputs.shape} (should be [batch_size, {n_classes}])")

if __name__ == "__main__":
    test_forward_on_medmnist(subset="pathmnist", batch_size=4)
    test_forward_on_medmnist(subset="chestmnist", batch_size=4)


Using downloaded and verified file: /home/vito/.medmnist/pathmnist.npz
Using downloaded and verified file: /home/vito/.medmnist/pathmnist.npz


KeyError: 'n_classes'

## Partial Train and Evaluate

In [ ]:
import torch
from src.utils import load_medmnist, evaluate_model
from src.models import InceptionMNISTModel
from src.trainer import Trainer

# Use batch_size=2 to reduce memory pressure
def check_partial_train(subset="pathmnist", batch_size=2):
    train_loader, val_loader, test_loader, n_classes, n_channels = load_medmnist(
        batch_size=batch_size,
        subset=subset,
        validation_split=0.2,
        shuffle_dataset=False,
        random_seed=42
    )

    print(f"\n---- Partial‐train on '{subset}' (batch_size={batch_size}) ----")
    print(f"Classes: {n_classes}, Channels: {n_channels}")

    branches = [{
        "depth": 1,
        "filter_sizes": [(3, 3)],
        "filter_channels": [8],
        "use_pooling": False
    }]
    model = InceptionMNISTModel(
        model_params={"branches_params": branches},
        input_channels=n_channels,
        num_classes=n_classes
    )

    cfg = {
        "learning_rate": 1e-3,
        "num_batches": 1,     # just one batch
        "num_epochs": 1,
        "patience": 1,
        "checkpoints_dir": "./tmp_ckpt"
    }
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    trainer = Trainer(device, cfg)
    model.to(device)

    # Run one‐batch partial training
    trainer.partial_train(model, train_loader)
    print("Partial train (1 batch) complete.")

    # Evaluate on validation set
    fitness, accuracy, loss = evaluate_model(
        model, val_loader, device,
        fitness_method="linear", alpha=3, beta=1.0
    )
    print(f"Post‐partial_train → Fitness: {fitness:.4f}, Accuracy: {accuracy:.2f}%, Loss: {loss:.4f}")

# Check on single-label subset
check_partial_train(subset="pathmnist", batch_size=2)
# Check on multi-label subset (if available)
check_partial_train(subset="chestmnist", batch_size=2)